# 01 — Read My Ears replication

**Cel:** uruchomić ear-movement detector z `vendor/read-my-ears` (Alves et al. 2025, CVPR Workshop) na ich własnym przykładowym wideo i wygenerować wykres timeline ruchu uszu.

**Wynik:** `../outputs/ear_movement_timeline.png` + ewentualnie annotated MP4.

**Gate:** punkt 3 z `../GATE.md`.

**Uwaga:** repo Read My Ears jest research-grade. API może wymagać dostosowania. Cell 1 pokazuje strukturę repo — jeśli inna niż oczekujemy, dostosować ścieżki w Cell 2.

## 1. Sprawdź strukturę vendor/read-my-ears

In [ ]:
import os, subprocess
from pathlib import Path

POC = Path(os.getcwd()).resolve()
if POC.name == 'notebooks':
    POC = POC.parent
os.chdir(POC)
RME = POC / 'vendor' / 'read-my-ears'
assert RME.exists(), f'Brak {RME} — uruchom bash setup.sh'

print('=== Top-level repo ===')
for p in sorted(RME.iterdir()):
    print(('DIR ' if p.is_dir() else 'FIL '), p.name)

readme = RME / 'README.md'
if readme.exists():
    print('\n=== README (pierwsze 80 linii) ===')
    print('\n'.join(readme.read_text(errors='ignore').splitlines()[:80]))

## 2. Inferencja na ich sample video

Repo struktury folderów: `VideoMAE-LSTM/`, `i3d-LSTM/`, `videomae_finetune_ears/`. Każdy ma własne `inference.py` lub `predict.py`. Strategie:

1. **Najprostsze**: znajdź ich sample video w repo (`examples/`, `data/`, `assets/`) i odpal ich `inference.py` przez `subprocess.run`.
2. **Fallback**: jeśli brak sample video — użyj `../data/sample_horse.mp4` (z notebooka 00) jako wejście.

Wybór skryptu: zaczynamy od `videomae_finetune_ears` (najnowszy backbone).

In [ ]:
# Lokalizacja sample video w repo Read My Ears
sample_candidates = (
    list(RME.rglob('*.mp4'))
    + list(RME.rglob('*.MP4'))
    + list(RME.rglob('*.mov'))
)
if sample_candidates:
    print('Znalezione video w repo Read My Ears:')
    for v in sample_candidates[:10]:
        print(f'  {v.relative_to(RME)}  ({v.stat().st_size/1e6:.1f} MB)')
    rme_sample = sample_candidates[0]
else:
    print('Brak własnego sample video w repo Read My Ears.')
    rme_sample = POC / 'data' / 'sample_horse.mp4'
    print(f'Fallback: użyję {rme_sample}')

In [ ]:
# Lokalizacja skryptu inference
candidates = (
    list(RME.rglob('inference*.py'))
    + list(RME.rglob('predict*.py'))
    + list(RME.rglob('demo*.py'))
    + list(RME.rglob('test*.py'))
)
print('Skrypty kandydaci:')
for s in candidates[:20]:
    print(f'  {s.relative_to(RME)}')

if not candidates:
    raise RuntimeError(
        'Brak skryptu inference w vendor/read-my-ears. '
        'Sprawdź README repo i dostosuj ten notebook ręcznie.'
    )

In [ ]:
# Próba uruchomienia inference. UWAGA: większość research-grade repos wymaga ręcznego dostrojenia ścieżek.
# Jeśli ten cell padnie z błędem, otwórz README z cell 1 i odpal CLI ręcznie.
import sys
script = candidates[0]
print(f'Uruchamiam: {script.relative_to(RME)}')
result = subprocess.run(
    [sys.executable, str(script), '--video', str(rme_sample)],
    capture_output=True, text=True, cwd=str(script.parent)
)
print('--- STDOUT (ostatnie 80 linii) ---')
print('\n'.join(result.stdout.splitlines()[-80:]))
if result.returncode != 0:
    print('--- STDERR (ostatnie 60 linii) ---')
    print('\n'.join(result.stderr.splitlines()[-60:]))
    print(f'\nReturncode {result.returncode}. To często znaczy że skrypt oczekuje innych argumentów.')
    print('Otwórz README repo (cell 1) i uruchom inference ręcznie z terminala.')

## 3. Heurystyczny wykres ruchu uszu (fallback z DLC keypoints)

Jeśli inferencja oryginalna z notatnika 01 cell 4 nie ruszyła w 2 h — wykorzystujemy keypoints z notatnika 00 (DLC SuperAnimal-Quadruped ma `left_earbase`, `right_earbase`, `left_eartip`, `right_eartip`) i liczymy proxy ruchu uszu jako rolling std pozycji w czasie.

To **nie** jest replikacja Read My Ears, ale daje sygnał czy w ogóle keypoints uszu są stabilne na sample clipie.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

out = POC / 'outputs'
h5_files = sorted(out.glob('*.h5'))
assert h5_files, f'Brak .h5 z keypoints w {out}. Najpierw uruchom 00_smoke_dlc_sample.ipynb.'
df = pd.read_hdf(h5_files[0])
print('Kolumny (top-level):', df.columns.get_level_values(-1).unique().tolist()[:30])

ear_parts = [c for c in df.columns.get_level_values(-2).unique() if 'ear' in c.lower()]
print('Wykryte ear keypoints:', ear_parts)

if ear_parts:
    fig, ax = plt.subplots(figsize=(10, 4))
    for part in ear_parts:
        cols = [c for c in df.columns if c[-2] == part and c[-1] in ('x', 'y')]
        if not cols:
            continue
        xy = df[cols].values
        movement = np.sqrt(np.sum(np.diff(xy, axis=0) ** 2, axis=1))
        rolling = pd.Series(movement).rolling(15, min_periods=1).mean()
        ax.plot(rolling, label=part)
    ax.set_xlabel('klatka')
    ax.set_ylabel('frame-to-frame ruch (px, rolling 15)')
    ax.set_title('Proxy ruchu uszu z DLC SuperAnimal keypoints (NIE replikacja Read My Ears)')
    ax.legend(loc='best')
    fig.tight_layout()
    fig.savefig(out / 'ear_movement_timeline.png', dpi=120)
    plt.show()
    print('Zapisano:', out / 'ear_movement_timeline.png')
else:
    print('Brak ear keypoints w wyjściu DLC (model sam zdecydował, że uszu nie widać).')

## 4. Notatka decyzyjna

**Punkt 3 z GATE.md:**
- TAK = oryginalny Read My Ears odpalił się i wygenerował wykres LUB fallback z DLC dał rozsądny `ear_movement_timeline.png`
- NIE = ani jedno ani drugie nie zadziałało

Wracaj do `../GATE.md` i wypełnij 4 punkty.